# Capstone 3 — Supervisor + 2 Workers on Earnings Transcripts (compressed, ~2.5h)

Spec: `README.md` in this folder. Build a supervisor + classifier + summariser over 3–5 earnings-call transcripts, plus a qualitative single-shot comparison.

LangChain cheatsheet: `../../langchain_langgraph.ipynb` §§2, 6, 9, 12.

**Time blocks**
| Block | Section | Goal |
|---|---|---|
| 0:00–0:20 | H1 | Load 3–5 transcripts |
| 0:20–0:45 | H2 | Worker schemas + agents (in isolation) |
| 0:45–1:15 | H3 | Supervisor + StateGraph |
| 1:15–1:45 | H4 | Run on all transcripts → result table |
| 1:45–2:15 | H5 | Single-shot comparison (qualitative) |
| 2:15–2:30 | H6 | Notes |

## Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

HERE = Path.cwd()
DATA = HERE / "data"

os.environ.setdefault("LANGSMITH_PROJECT", "capstone-3-supervisor")

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing"
assert ".venv" in sys.executable, f"Wrong kernel: {sys.executable}"
print("python:", sys.executable)
print("data  :", DATA, "exists:", DATA.exists())

---
## H1 — Load 3–5 transcripts (0:00–0:20)

**Hints**
- Quickest source: HuggingFace. `from datasets import load_dataset; ds = load_dataset('jlh-ibm/earnings_call', split='train')` — returns a `Dataset` of dicts.
  - If that one isn't ideal, try `lamini/earnings-calls-qa` or search HF for `earnings call transcript`.
- **Trim each transcript to ~6k tokens** (≈24k chars) to keep API calls cheap. Use the first 24k chars for v1.
- Persist to `data/transcripts/{ticker}.txt` so re-runs don't re-download.
- For each transcript, capture `{ticker, period, transcript_text}`.
- **Sanity**: 3–5 transcripts loaded, each 5k–25k chars after trimming.

In [ ]:
# from datasets import load_dataset

# --- TODO ---
# 1) load_dataset(...), pick 3–5 rows
# 2) for each row, build a dict: {"ticker": ..., "period": ..., "transcript": ...[:24000]}
# 3) persist to data/transcripts/<ticker>.txt (idempotent)
# 4) return list[dict]

TRANSCRIPTS: list[dict] = ...  # your loader output

# Sanity
# for t in TRANSCRIPTS:
#     print(f"{t['ticker']:>6s} {t['period']:>10s}  {len(t['transcript']):>6d} chars")

---
## H2 — Worker schemas + agents (0:20–0:45)

Two workers, each with structured output. Test in isolation on 1 transcript before wiring the supervisor.

**Hints**
- Cheatsheet §2 — `init_chat_model('openai:gpt-4o-mini').with_structured_output(YourSchema)`.
- Keep each worker's system prompt short; the LLM does not benefit from a 500-word prompt.
- For the summariser, instruct the model to include ONE direct quote — that's what the (cut) faithfulness evaluator would later check.

In [ ]:
# from typing import Literal
# from pydantic import BaseModel, Field
# from langchain.chat_models import init_chat_model

# class Classification(BaseModel):
#     guidance: Literal["beat", "miss", "inline"]
#     sentiment: Literal["positive", "neutral", "negative"]
#     reasoning: str = Field(description="One short sentence of justification.")

# class Summary(BaseModel):
#     summary: str = Field(description="~150 words.")
#     key_quote: str = Field(description="One direct quote pulled verbatim from the transcript.")

# llm_cheap = init_chat_model("openai:gpt-4o-mini", temperature=0)

# def classify_node(state: "State") -> dict:
#     model = llm_cheap.with_structured_output(Classification)
#     out = model.invoke(f"Classify this earnings call:\n\n{state['transcript']}")
#     return {"classification": out, "next": ...}  # supervisor will overwrite next

# def summarise_node(state: "State") -> dict:
#     model = llm_cheap.with_structured_output(Summary)
#     out = model.invoke(f"Summarise this call in ~150 words and pull one direct quote:\n\n{state['transcript']}")
#     return {"summary": out}

classify_node = ...  # your worker
summarise_node = ...  # your worker

In [ ]:
# Sanity — call each worker on transcript[0] in isolation
# t0 = TRANSCRIPTS[0]
# print("classifier:", classify_node({"transcript": t0["transcript"]}))
# print("summariser:", summarise_node({"transcript": t0["transcript"]}))

---
## H3 — Supervisor + StateGraph (0:45–1:15)

**Hints**
- Cheatsheet §6, §9.
- Shared state schema (TypedDict):
  ```python
  class State(TypedDict):
      transcript: str
      classification: Optional[Classification]
      summary: Optional[Summary]
      next: str
      step: int       # safety counter; supervisor returns 'finish' if step > 4
  ```
- Supervisor logic (no LLM call needed for this — deterministic routing is fine for 2 workers):
  - if `classification is None`: return `{'next': 'classifier'}`
  - elif `summary is None`: return `{'next': 'summariser'}`
  - else: return `{'next': 'finish'}`
  - For the *interview talking point*, you could swap deterministic routing for an LLM router with `with_structured_output(Router)` — but it adds cost without value at 2 workers. Note this in your end-of-day notes.
- `add_conditional_edges('supervisor', lambda s: s['next'], {'classifier': 'classifier', 'summariser': 'summariser', 'finish': END})`.

In [ ]:
# from typing import Optional, TypedDict
# from langgraph.graph import StateGraph, START, END

# class State(TypedDict):
#     transcript: str
#     classification: Optional[Classification]
#     summary: Optional[Summary]
#     next: str
#     step: int

# def supervisor(state: State) -> dict:
#     ...

# graph = StateGraph(State)
# graph.add_node("supervisor", supervisor)
# graph.add_node("classifier", classify_node)
# graph.add_node("summariser", summarise_node)
# graph.add_edge(START, "supervisor")
# graph.add_conditional_edges("supervisor", lambda s: s["next"],
#                              {"classifier": "classifier",
#                               "summariser": "summariser",
#                               "finish": END})
# graph.add_edge("classifier", "supervisor")
# graph.add_edge("summariser", "supervisor")
# app = graph.compile()

app = ...  # your compiled graph

---
## H4 — Run on all transcripts → result table (1:15–1:45)

In [ ]:
# import pandas as pd

# rows = []
# for t in TRANSCRIPTS:
#     config = {"tags": ["capstone-3", "multi-agent"], "metadata": {"ticker": t["ticker"]}}
#     final = app.invoke(
#         {"transcript": t["transcript"], "classification": None, "summary": None, "next": "", "step": 0},
#         config=config,
#     )
#     rows.append({
#         "ticker": t["ticker"],
#         "period": t["period"],
#         "guidance": final["classification"].guidance,
#         "sentiment": final["classification"].sentiment,
#         "summary": final["summary"].summary[:200] + "...",
#     })
# pd.DataFrame(rows)

---
## H5 — Single-shot baseline (1:45–2:15) — qualitative

**Hints**
- One prompt that asks for both fields together. Pydantic schema = union of `Classification` + `Summary`.
- Run on the same transcripts; tag traces `single-shot` (vs `multi-agent` above).
- **Don't formally evaluate** — just eyeball ONE example side-by-side.
- The likely finding: single-shot is ~half the cost and within a few tokens on quality. Multi-agent isn't free; for this task it probably doesn't earn its overhead.
- Note the finding in your end-of-day cell — "for this task I'd ship single-shot" is a strong interview answer.

In [ ]:
# class CombinedOutput(BaseModel):
#     guidance: Literal["beat", "miss", "inline"]
#     sentiment: Literal["positive", "neutral", "negative"]
#     summary: str
#     key_quote: str

# single_shot_llm = llm_cheap.with_structured_output(CombinedOutput)

# t0 = TRANSCRIPTS[0]
# config = {"tags": ["capstone-3", "single-shot"], "metadata": {"ticker": t0["ticker"]}}
# single_shot_out = single_shot_llm.invoke(
#     f"Classify guidance + sentiment AND summarise in ~150 words with one direct quote:\n\n{t0['transcript']}",
#     config=config,
# )
# print(single_shot_out)

---
## H6 — Notes (2:15–2:30)

Fill in once you've eyeballed the comparison.

- **Three LangSmith trace links** (one multi-agent, one single-shot, one of your choice):
  1. …
  2. …
  3. …
- **Quality side-by-side** on transcript 0: which output looked better? On what dimension?
- **Cost note**: roughly how many LLM calls did multi-agent take vs single-shot? (LangSmith shows the count per trace.)
- **Which would I ship?**: …  *(most likely "single-shot for this task" — that's the interview-defensible answer)*
- **What I'd add for production**: third worker (risk-flag scanner), 30-call hand-labelled eval set with classification F1 + faithfulness LLM-judge, prompt caching, cost-engineered model fallback. (See `README.md` → "What got cut".)